# WebQSP with Neo4j-backed graph input

This example keeps the complete WebQSP graph in an already-populated Neo4j database. FastRLM gives the graph child a bounded read-only Cypher operation; every query result is returned to it as a NetworkX `MultiDiGraph`. Set `NEO4J_PASSWORD` and `RLM_MODEL` in the environment before running, and use a Neo4j account with server-enforced read-only privileges.

Entity mentions are resolved through the pre-created `webqsp_entity_names` full-text index. For example, the child turns `Ben Franklin` into the parameterized Lucene query `ben* AND franklin*`, then reranks candidates using matching relationship predicates before retrieving answer evidence.

In [7]:
import os
from statistics import median
from typing import Literal
import json

import fast_rlm
from datasets import load_dataset
from fast_rlm import Neo4jGraph, RLMConfig
from pydantic import BaseModel

/Users/kapish/Kapish/RP/fast-rlm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Constants: change the question/example pair together.
NEO4J_URI = os.getenv("NEO4J_URI", "neo4j://localhost:7687")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j-admin-2026")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")
FULLTEXT_INDEX = "webqsp_entity_names"
RLM_MODEL = os.getenv("RLM_MODEL", "poolside/laguna-s-2.1:free")
# os.environ.setdefault("RLM_MODEL_BASE_URL", "http://localhost:11434/v1")
# os.environ.setdefault("RLM_MODEL_API_KEY", "ollama")
QUESTION = "what does jamaican people speak"
Q_ENTITY = ["Jamaica"]
GOLD_ANSWERS = ["Jamaican English", "Jamaican Creole English Language"]
WEBQSP_SCHEMA = """
(:WebQSPExample {id, source_id, question, split, q_entity_json})
(:WebQSPEntity {key, name})
(:WebQSPEntity)-[:RELATED {predicate}]->(:WebQSPEntity)
""".strip() + f"\nFULLTEXT INDEX {FULLTEXT_INDEX} FOR (:WebQSPEntity) ON (name)"

In [9]:
print(RLM_MODEL)

deepseek/deepseek-v4.1-flash


In [10]:
class WebQSPAnswer(BaseModel):
    answer_entities: list[str]
    status: Literal["answered", "insufficient_data"]


graph = Neo4jGraph(
    uri=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
    schema=WEBQSP_SCHEMA,
)
config = RLMConfig(
    primary_agent=RLM_MODEL,
    sub_agent=RLM_MODEL,
    max_depth=3,
    max_graph_queries=1,
    max_graph_hops=4,
    max_cypher_queries=5,
    max_cypher_rows=100,
    cypher_timeout_ms=15_000,
)

RUN_INSTRUCTION = (
    f"Delegate graph retrieval once. Resolve complete question entity mentions through "
    f"the {FULLTEXT_INDEX} full-text index using a parameterized Lucene prefix query. "
    "Rerank candidates with their incident relationship predicates, then scope answer "
    "queries to selected stable key values. Return only answer-supporting evidence."
)

In [6]:
result = fast_rlm.run(
    {
        "question": QUESTION,
    },
    graph=graph,
    config=config,
    output_schema=WebQSPAnswer,
    evaluation_q_entity=Q_ENTITY,
    instruction=RUN_INSTRUCTION,
    # verbosity="summary",
)

answer = result["results"]
with open(result["log_file"]) as log_handle:
    events = [json.loads(line) for line in log_handle if line.strip()]
audit_events = [
    event for event in events
    if event.get("event_type", "").startswith(("graph_", "neo4j_"))
]
model_calls = sum(
    event.get("usage", {}).get("total_tokens", 0) > 0
    for event in events
)
entity_evaluation = next(
    (event for event in reversed(audit_events) if "entity_precision" in event),
    {},
)

✔ Neo4j connected: database=neo4j
📝 Logging to: /Users/kapish/Kapish/RP/fast-rlm/examples/logs/run_2026-10-01T16-29-38-778Z.jsonl

✔ Python Ready
✔ requests + httpx ready (env setup took 1277ms)


⠋ Generating code...
✔ Code generated


↳ graph_query called
✔ Python Ready
✔ requests + httpx ready (env setup took 1037ms)


⠋ Generating code...
✔ Code generated
⠋ Generating code...


    │ ────────────────────────── Depth 1 · Step 1/20 ──────────────────────────
    │ ╭ Reasoning ─────────────────────────────────────────────────────────────╮
    │ │ We need answer question: "what does jamaican people speak" likely      │
    │ │ language of Jamaica. Need retrieve evidence from WebQSP graph. Need    │
    │ │ start with fulltext search on entity mention from question. Question   │
    │ │ mentions "jamaican people" maybe entity "Jamaican people"? Need query  │
    │ │ fulltext index webqsp_entity_names for complete entity mention. Use    │
    │ │ build_fulltext_entity_query helper. Let's do a Cypher call using       │
    │ │ db.index.fulltext.queryNodes. We need only one validated Cypher        │
    │ │ statement in this step. We can call helper to build query. Need use    │
    │ │ parameters. Let's inspect helper? We can call                          │
    │ │ build_fulltext_entity_query("jamaican people") and use as parameter.   │
    │ │ But we need do one Cy

...

⠋ Generating code...
✔ Code generated
⠋ Generating code...


    │ ────────────────────────── Depth 1 · Step 4/20 ──────────────────────────
    │ ╭ Reasoning ─────────────────────────────────────────────────────────────╮
    │ │ The question is "what does jamaican people speak" - the answer is      │
    │ │ language spoken. I found "Jamaican Creole English Language" and likely │
    │ │ "English Language" and "Jamaican Patois". Let me look for language     │
    │ │ predicate on Jamaica.                                                  │
    │ │                                                                        │
    │ │ Let me query languages spoken in Jamaica with specific predicate.      │
    │ ╰────────────────────────────────────────────────────────────────────────╯
    │ ╭ Python Code ───────────────────────────────────────────────────────────╮
    │ │ 1 result = await execute_read_only_cypher(                             │
    │ │ 2     """                                                              │
    │ │ 3     MATCH (a:WebQSP

✔ Code generated
⠋ Generating code...


    │ ────────────────────────── Depth 1 · Step 5/20 ──────────────────────────
    │ ╭ Reasoning ─────────────────────────────────────────────────────────────╮
    │ │ The question: "what does jamaican people speak" - answer: Jamaican     │
    │ │ English, Jamaican Creole English. We have evidence. Let me finalize.   │
    │ │                                                                        │
    │ │ Relevant predicates: location.country.official_language,               │
    │ │ location.country.languages_spoken. Source name: Jamaica.               │
    │ ╰────────────────────────────────────────────────────────────────────────╯
    │ ╭ Python Code ───────────────────────────────────────────────────────────╮
    │ │ 1 relevant_predicates = {                                              │
    │ │ 2     "location.country.official_language",                            │
    │ │ 3     "location.country.languages_spoken",                             │
    │ │ 4 }                  

✔ Code generated


In [ ]:
print("Question:", QUESTION)
print("Prediction:", answer["answer_entities"])
print("Status:", answer["status"])
print("Golden answer:", GOLD_ANSWERS)
print("Entity precision:", entity_evaluation.get("entity_precision"))
print("Entity recall:", entity_evaluation.get("entity_recall"))
print("Seed entities considered:", entity_evaluation.get("seed_entities_considered", []))
print("Selected seed entities:", entity_evaluation.get("selected_seed_entities", []))
print("Model calls:", model_calls)
print("Total tokens:", result["usage"]["total_tokens"])
print("Graph-native audit:")
for event in audit_events:
    fields = []
    details = event.get("graph", {})
    if details:
        fields.append(
            f"graph={details.get('backend')}/{details.get('graph_type', 'source')} "
            f"nodes={details.get('nodes', '-')}, edges={details.get('edges', '-')}"
        )
    for key in (
        "cypher_query_number", "parameter_names", "row_limit", "duration_ms",
        "rows", "truncated", "max_graph_hops", "entity_mentions",
        "candidate_count", "candidates", "min_entity_confidence",
        "entity_precision", "entity_recall", "seed_source", "seed_count",
        "seed_entities_considered", "selected_seed_entities",
        "seeds", "seed_match_methods", "error", "error_code",
    ):
        if key in event:
            fields.append(f"{key}={event[key]}")
    print(f"  {event['event_type']}: " + ", ".join(fields))

## Ten-question WebQSP test-set evaluation

This section runs the first ten examples from the deterministic `rmanluo/RoG-webqsp` test split against the same Neo4j database and FastRLM configuration. Runs are sequential and use silent verbosity; each still writes its normal JSONL audit log.

In [11]:
DATASET_NAME = "rmanluo/RoG-webqsp"
BATCH_LIMIT = 10
test_rows = load_dataset(DATASET_NAME, split="test").select(range(BATCH_LIMIT))
print(f"Loaded {len(test_rows)} questions from {DATASET_NAME} test split.")

Loaded 10 questions from rmanluo/RoG-webqsp test split.


In [12]:
def normalized_answer_set(values):
    return {str(value).strip().casefold() for value in values if str(value).strip()}


def answer_f1(predicted, gold):
    predicted_set = normalized_answer_set(predicted)
    gold_set = normalized_answer_set(gold)
    if not predicted_set and not gold_set:
        return 1.0
    overlap = len(predicted_set & gold_set)
    precision = overlap / len(predicted_set) if predicted_set else 0.0
    recall = overlap / len(gold_set) if gold_set else 0.0
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0


batch_results = []
for index, row in enumerate(test_rows, start=1):
    question = str(row["question"])
    q_entity = [str(entity) for entity in row["q_entity"]]
    gold_answers = [str(answer) for answer in row["answer"]]
    try:
        batch_run = fast_rlm.run(
            {"question": question},
            graph=graph,
            config=config,
            output_schema=WebQSPAnswer,
            evaluation_q_entity=q_entity,
            instruction=RUN_INSTRUCTION,
            verbosity="silent",
        )
        prediction = [str(value) for value in batch_run["results"]["answer_entities"]]
        status = str(batch_run["results"]["status"])
        total_tokens = int(batch_run["usage"]["total_tokens"])
        score = answer_f1(prediction, gold_answers)
        exact_match = normalized_answer_set(prediction) == normalized_answer_set(gold_answers)
        record = {
            "index": index,
            "question": question,
            "prediction": prediction,
            "gold_answers": gold_answers,
            "status": status,
            "answer_f1": score,
            "exact_match": exact_match,
            "total_tokens": total_tokens,
        }
    except Exception as error:
        record = {
            "index": index,
            "question": question,
            "prediction": [],
            "gold_answers": gold_answers,
            "status": "run_error",
            "answer_f1": 0.0,
            "exact_match": False,
            "total_tokens": 0,
            "error_type": type(error).__name__,
        }
    batch_results.append(record)
    print(
        f"[{index:02d}/{BATCH_LIMIT}] status={record['status']} "
        f"exact={record['exact_match']} f1={record['answer_f1']:.3f} "
        f"tokens={record['total_tokens']}"
    )
    print("  Question:", question)
    print("  Prediction:", record["prediction"])
    print("  Gold:", gold_answers)
    if "error_type" in record:
        print("  Error type:", record["error_type"])

[01/10] status=answered exact=True f1=1.000 tokens=17224
  Question: what does jamaican people speak
  Prediction: ['Jamaican Creole English Language', 'Jamaican English']
  Gold: ['Jamaican English', 'Jamaican Creole English Language']
[02/10] status=answered exact=False f1=0.500 tokens=26970
  Question: what did james k polk do before he was president
  Prediction: ['United States Representative']
  Gold: ['United States Representative', 'Governor of Tennessee', 'Speaker of the United States House of Representatives']
[03/10] status=insufficient_data exact=False f1=0.000 tokens=21122
  Question: who plays ken barlow in coronation street
  Prediction: []
  Gold: ['William Roache']
[04/10] status=answered exact=True f1=1.000 tokens=19626
  Question: where is jamarcus russell from
  Prediction: ['Mobile']
  Gold: ['Mobile']
[05/10] status=answered exact=True f1=1.000 tokens=22593
  Question: where was george washington carver from
  Prediction: ['Diamond']
  Gold: ['Diamond']
[06/10] st

In [13]:
completed_results = [row for row in batch_results if row["status"] != "run_error"]
token_counts = [row["total_tokens"] for row in completed_results]
print("Completed runs:", len(completed_results), "/", BATCH_LIMIT)
print("Answered runs:", sum(row["status"] == "answered" for row in completed_results))
print("Exact matches:", sum(row["exact_match"] for row in completed_results))
print(
    "Macro answer F1:",
    round(sum(row["answer_f1"] for row in completed_results) / len(completed_results), 3)
    if completed_results else 0.0,
)
print("Total tokens:", sum(token_counts))
print("Median tokens per completed run:", int(median(token_counts)) if token_counts else 0)

Completed runs: 10 / 10
Answered runs: 8
Exact matches: 5
Macro answer F1: 0.59
Total tokens: 232344
Median tokens per completed run: 21427
